# POCD-KansformerEPI v3 — Fix & Deploy

**What this fixes:**
- `feats_config` path — JSON is in `genomic_data/`, NOT `genomic_data/processed/`
- `nn.Sigmoid()` removed from classifier (using `BCEWithLogitsLoss`)
- Weighted sampling + `pos_weight` to handle class imbalance
- Data augmentation (reverse complement, epi noise, bin shift)

Run each cell in order.

## Cell 1 — Stop previous training & set directory

In [ ]:
import os, subprocess
subprocess.run(['pkill', '-f', 'python train.py'], capture_output=True)
os.chdir(os.path.expanduser('~/POCD-KansformerEPI'))
print(f"Working dir: {os.getcwd()}")
print("Previous training stopped.")

## Cell 2 — Diagnose all data paths

Checks every file the pipeline needs and reports issues.

In [ ]:
import os, json, glob

ok_count, fail_count = 0, 0

def check(label, path):
    global ok_count, fail_count
    exists = os.path.exists(path)
    status = 'OK' if exists else 'MISSING'
    if exists:
        ok_count += 1
    else:
        fail_count += 1
    size = ''
    if exists and os.path.isfile(path):
        mb = os.path.getsize(path) / (1024**2)
        size = f' ({mb:.1f} MB)'
    print(f"  [{status}] {label}: {path}{size}")
    return exists

print('=' * 60)
print('DATA PATH DIAGNOSIS')
print('=' * 60)

# 1. BENGI files
print('\n--- BENGI TSV files ---')
bengi_dir = './data/BENGI'
check('BENGI directory', bengi_dir)
bengi_files = sorted(glob.glob(os.path.join(bengi_dir, '*.tsv.gz')) + glob.glob(os.path.join(bengi_dir, '*.tsv')))
print(f"  Found {len(bengi_files)} BENGI files")
for f in bengi_files:
    print(f"    {os.path.basename(f)}")

# 2. JSON config (try all candidates)
print('\n--- Feature config JSON ---')
json_candidates = [
    './data/genomic_data/CTCF_DNase_6histone_local.500.json',
    './data/genomic_data/processed/CTCF_DNase_6histone_local.500.json',
]
found_json = None
for j in json_candidates:
    if check(os.path.basename(j), j):
        if found_json is None:
            found_json = j

# Also look for any .json files in data/genomic_data
other_jsons = glob.glob('./data/genomic_data/*.json')
if other_jsons:
    print(f"  All JSON files found: {[os.path.basename(j) for j in other_jsons]}")

# 3. Show JSON contents
if found_json:
    print(f'\n--- Contents of {found_json} ---')
    with open(found_json) as fh:
        jdata = json.load(fh)
    # Show raw contents (first 2000 chars)
    print(json.dumps(jdata, indent=2)[:2000])

    # 4. Check if .pt paths resolve correctly
    print(f'\n--- Resolving .pt paths ---')
    jdata_copy = dict(jdata)  # work on copy
    base = jdata_copy.pop('_location', None)
    if base is None:
        base = os.path.dirname(os.path.abspath(found_json))
    print(f'  Base location: {base}')
    pt_ok, pt_fail = 0, 0
    for cell, assays in jdata_copy.items():
        if not isinstance(assays, dict):
            continue
        for mark, fn in assays.items():
            if not isinstance(fn, str):
                continue
            full = fn if os.path.isabs(fn) else os.path.join(base, fn)
            if os.path.exists(full):
                pt_ok += 1
            else:
                pt_fail += 1
                print(f'  MISSING: {cell}/{mark} -> {full}')
    print(f'  .pt files: {pt_ok} found, {pt_fail} missing')
else:
    print('\n  WARNING: No JSON config found at all!')

# 5. hg19
print('\n--- Reference genome ---')
hg19_path = os.path.expanduser('~/POCD-KansformerEPI/data/reference/hg19.fa')
check('hg19.fa', hg19_path)
check('hg19.fa.fai', hg19_path + '.fai')

# 6. pysam
print('\n--- pysam ---')
try:
    import pysam
    print(f'  [OK] pysam {pysam.__version__}')
except ImportError:
    print('  [MISSING] pysam not installed! Run: pip install pysam')
    fail_count += 1

# 7. .pt files
print('\n--- .pt feature files ---')
pt_files = glob.glob('./data/genomic_data/processed/*.pt')
print(f'  Found {len(pt_files)} .pt files in data/genomic_data/processed/')

print(f"\n{'='*60}")
print(f'RESULT: {ok_count} OK, {fail_count} MISSING')
if fail_count > 0:
    print('FIX the missing items before continuing!')
else:
    print('All paths verified.')

## Cell 3 — Rebuild JSON config (run ONLY if Cell 2 shows missing .pt files)

This scans the actual `.pt` files on disk and creates a correct JSON config.
**Skip this cell if Cell 2 shows all .pt files resolved correctly.**

In [ ]:
%%writefile configs/config.yaml
experiment_name: "POCD_Kansformer_v3"

paths:
  save_dir: "./checkpoints"
  bengi_dir: "./data/BENGI"
  feats_config: "./data/genomic_data/CTCF_DNase_6histone_local.500.json"
  ref_genome: "/home/jupyter/POCD-KansformerEPI/data/reference/hg19.fa"

data:
  sequence_length: 6000
  epigenetic_bins: 5000
  n_epigenetic_features: 8
  kmer_size: 3
  batch_size: 128
  bin_size: 500
  seq_len_bp: 2500000
  enhancer_window: 3000
  promoter_window: 3000
  feats_order:
    - "CTCF"
    - "DNase"
    - "H3K27ac"
    - "H3K27me3"
    - "H3K36me3"
    - "H3K4me1"
    - "H3K4me3"
    - "H3K9me3"

model:
  hidden_dim: 256
  num_heads: 8
  num_layers: 4
  n_tokens: 128
  dropout: 0.3

training:
  lr: 0.00005
  epochs: 50
  lambda_dist: 0.1
  grad_clip: 1.0
  patience: 15
  num_workers: 8
  weight_decay: 0.001
  valid_chroms:
    - "chr11"
    - "chr17"

augmentation:
  enabled: true
  rc_prob: 0.5
  epi_noise_std: 0.05
  shift_max_bins: 3

## Cell 4 — Install pysam (if missing)

In [ ]:
try:
    import pysam
    print(f'pysam already installed: {pysam.__version__}')
except ImportError:
    print('Installing pysam...')
    !pip install pysam -q
    import pysam
    print(f'pysam installed: {pysam.__version__}')

## Cell 5 — Write config.yaml (fixed `feats_config` path + batch_size 128)

In [ ]:
%%writefile configs/config.yaml
experiment_name: "POCD_Kansformer_v3"

paths:
  save_dir: "./checkpoints"
  bengi_dir: "./data/BENGI"
  feats_config: "./data/genomic_data/CTCF_DNase_6histone_local.500.json"
  ref_genome: "/home/jupyter/POCD-KansformerEPI/data/reference/hg19.fa"

data:
  sequence_length: 6000
  epigenetic_bins: 5000
  n_epigenetic_features: 8
  kmer_size: 3
  batch_size: 128
  bin_size: 500
  seq_len_bp: 2500000
  enhancer_window: 3000
  promoter_window: 3000
  feats_order:
    - "CTCF"
    - "DNase"
    - "H3K27ac"
    - "H3K27me3"
    - "H3K36me3"
    - "H3K4me1"
    - "H3K4me3"
    - "H3K9me3"

model:
  hidden_dim: 256
  num_heads: 8
  num_layers: 4
  n_tokens: 128
  dropout: 0.3

training:
  lr: 0.00005
  epochs: 50
  lambda_dist: 0.1
  grad_clip: 1.0
  patience: 15
  num_workers: 8
  weight_decay: 0.001
  valid_chroms:
    - "chr11"
    - "chr17"

augmentation:
  enabled: true
  rc_prob: 0.5
  epi_noise_std: 0.05
  shift_max_bins: 3

## Cell 6 — Fix model.py (remove Sigmoid)

In [ ]:
with open('src/model.py', 'r') as f:
    code = f.read()

if 'nn.Sigmoid()' in code:
    code = code.replace('            nn.Sigmoid()\n', '')
    with open('src/model.py', 'w') as f:
        f.write(code)
    print('Removed nn.Sigmoid() from classifier')
else:
    print('nn.Sigmoid() already removed -- no change needed')

# Verify
with open('src/model.py', 'r') as f:
    for i, line in enumerate(f, 1):
        if 'Sigmoid' in line:
            print(f'  WARNING: Sigmoid still at line {i}: {line.strip()}')
            break
    else:
        print('  Verified: no Sigmoid in model.py')

## Cell 7 — Write dataset.py (with augmentation)

In [ ]:
%%writefile src/dataset.py
"""PyTorch Dataset wrapper that applies POCD-ND encoding on-the-fly."""

import torch
from torch.utils.data import Dataset
import numpy as np

_COMP = str.maketrans('ACGTNacgtn', 'TGCANtgcan')

def _reverse_complement(seq: str) -> str:
    """Return reverse complement of a DNA string."""
    return seq.translate(_COMP)[::-1]


class EPIDataset(Dataset):
    """
    Wrapper that applies POCD-ND encoding to raw DNA sequences.

    Each __getitem__ returns:
        seq   : Tensor (64, L)
        epi   : Tensor (n_feats, n_bins)
        label : Tensor (1,)
        dist  : Tensor (1,)
    """

    def __init__(self, config, encoder, source_dataset=None, sequences=None,
                 epi_features=None, labels=None, distances=None,
                 num_synthetic=0, concat_enh_prom=True):
        self.seq_len = config["data"]["sequence_length"]
        self.bins = config["data"]["epigenetic_bins"]
        self.n_epi = config["data"]["n_epigenetic_features"]
        self.encoder = encoder
        self.concat_enh_prom = concat_enh_prom

        # Augmentation settings (toggled per-phase in training loop)
        self.augment = False
        aug_cfg = config.get("augmentation", {})
        self.aug_rc_prob = aug_cfg.get("rc_prob", 0.5)
        self.aug_epi_noise_std = aug_cfg.get("epi_noise_std", 0.05)
        self.aug_shift_max = aug_cfg.get("shift_max_bins", 3)

        self.mode = None
        self._source = None
        self._manual_data = []

        if source_dataset is not None:
            self.mode = "pipeline"
            self._source = source_dataset
            print(f"EPIDataset: wrapping {len(self._source)} pipeline samples")
        elif sequences is not None:
            self.mode = "manual"
            self._build_manual(sequences, epi_features, labels, distances)
        elif num_synthetic > 0:
            self.mode = "synthetic"
            self._build_synthetic(num_synthetic)
        else:
            raise ValueError("Provide source_dataset, sequences, or num_synthetic > 0")

    def _build_manual(self, sequences, epi_features, labels, distances):
        n = len(sequences)
        if epi_features is None:
            epi_features = [np.zeros((self.n_epi, self.bins), dtype=np.float32) for _ in range(n)]
        if labels is None: labels = [0.0] * n
        if distances is None: distances = [0.0] * n
        for i in range(n):
            seq = self._pad_or_trim(sequences[i])
            epi = self._ensure_epi_shape(epi_features[i])
            self._manual_data.append((seq, epi, float(labels[i]), float(distances[i])))
        print(f"EPIDataset: {n} manual samples loaded")

    def _build_synthetic(self, n):
        bases = list("ACGT")
        for _ in range(n):
            seq = "".join(np.random.choice(bases, size=self.seq_len))
            epi = np.random.rand(self.n_epi, self.bins).astype(np.float32)
            label = float(np.random.randint(0, 2))
            dist = float(np.random.rand() * 10.0)
            self._manual_data.append((seq, epi, label, dist))
        print(f"EPIDataset: {n} synthetic samples generated")

    def _pad_or_trim(self, seq):
        if len(seq) > self.seq_len: return seq[:self.seq_len]
        if len(seq) < self.seq_len: return seq + "N" * (self.seq_len - len(seq))
        return seq

    def _ensure_epi_shape(self, epi):
        epi = np.asarray(epi, dtype=np.float32)
        if epi.shape == (self.n_epi, self.bins): return epi
        if epi.ndim == 2 and epi.shape == (self.bins, self.n_epi): return epi.T
        out = np.zeros((self.n_epi, self.bins), dtype=np.float32)
        r = min(epi.shape[0], self.n_epi)
        c = min(epi.shape[1] if epi.ndim > 1 else self.bins, self.bins)
        if epi.ndim == 1: out[0, :min(len(epi), self.bins)] = epi[:self.bins]
        else: out[:r, :c] = epi[:r, :c]
        return out

    def __len__(self):
        if self.mode == "pipeline": return len(self._source)
        return len(self._manual_data)

    def __getitem__(self, idx):
        if self.mode == "pipeline": return self._getitem_pipeline(idx)
        return self._getitem_manual(idx)

    def _getitem_pipeline(self, idx):
        """Retrieve from EPIGenomicDataset and apply POCD-ND encoding."""
        raw = self._source[idx]
        epi = raw["epi"].clone()

        if self.concat_enh_prom:
            dna = raw["enhancer_seq"] + raw["promoter_seq"]
        else:
            dna = raw["enhancer_seq"]
        dna = self._pad_or_trim(dna)

        # --- Data augmentation (training only) ---
        if self.augment:
            if np.random.random() < self.aug_rc_prob:
                dna = _reverse_complement(dna)
            if self.aug_epi_noise_std > 0:
                epi = epi + torch.randn_like(epi) * self.aug_epi_noise_std
                epi = epi.clamp(min=0)
            if self.aug_shift_max > 0:
                shift = np.random.randint(-self.aug_shift_max, self.aug_shift_max + 1)
                if shift != 0:
                    epi = torch.roll(epi, shifts=shift, dims=1)

        seq_enc = self.encoder.transform(dna)
        return {"seq": seq_enc, "epi": epi.float(), "label": raw["label"], "dist": raw["dist"]}

    def _getitem_manual(self, idx):
        seq, epi, label, dist = self._manual_data[idx]
        seq_enc = self.encoder.transform(seq)
        return {
            "seq": seq_enc,
            "epi": torch.tensor(epi, dtype=torch.float32),
            "label": torch.tensor([label], dtype=torch.float32),
            "dist": torch.tensor([dist], dtype=torch.float32),
        }

## Cell 8 — Write train.py (weighted loss + sampler + augmentation)

In [ ]:
%%writefile train.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset, WeightedRandomSampler
import yaml
import os
import glob
import numpy as np
import pickle
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupKFold

from src.epi_data_pipeline import EPIGenomicDataset
from src.dataset import EPIDataset
from src.model import Kansformer
from src.encoding import POCD_ND_Encoder
from src.visualize import plot_history

# 1. Setup
with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)
os.makedirs(config['paths']['save_dir'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 2. Prepare Encoder
print("Initializing POCD-ND Encoder...")
encoder = POCD_ND_Encoder(k=config['data']['kmer_size'])
np.random.seed(42)
dummy_pos = ["".join(np.random.choice(['A','C','G','T'], config['data']['sequence_length'])) for _ in range(50)]
dummy_neg = ["".join(np.random.choice(['A','C','G','T'], config['data']['sequence_length'])) for _ in range(50)]
encoder.fit(dummy_pos, dummy_neg, config['data']['sequence_length'])

with open(f"{config['paths']['save_dir']}/encoder.pkl", 'wb') as f:
    pickle.dump(encoder, f)
print("Encoder saved.")

# 3. Data Loading
bengi_dir = config['paths'].get('bengi_dir', './data/BENGI')
feats_config = config['paths'].get('feats_config', '')
ref_genome = config['paths'].get('ref_genome', '')

bengi_files = sorted(
    glob.glob(os.path.join(bengi_dir, '*.tsv.gz')) +
    glob.glob(os.path.join(bengi_dir, '*.tsv'))
)

USE_REAL_DATA = len(bengi_files) > 0 and os.path.exists(feats_config)

if USE_REAL_DATA:
    print(f"\n=== REAL DATA MODE ===")
    print(f"BENGI files: {len(bengi_files)}")
    for f_path in bengi_files:
        print(f"  {os.path.basename(f_path)}")

    genomic_ds = EPIGenomicDataset(
        bengi_paths=bengi_files,
        feats_config_path=feats_config,
        feats_order=config['data'].get('feats_order', None),
        seq_len=config['data'].get('seq_len_bp', 2_500_000),
        bin_size=config['data'].get('bin_size', 500),
        enhancer_window=config['data'].get('enhancer_window', 3000),
        promoter_window=config['data'].get('promoter_window', 3000),
        ref_genome_path=ref_genome if ref_genome else None,
    )

    dataset = EPIDataset(config, encoder, source_dataset=genomic_ds)

    valid_chroms = config['training'].get('valid_chroms', ['chr11', 'chr17'])
    chroms = genomic_ds.get_chrom_groups()
    train_idx = [i for i, c in enumerate(chroms) if c not in valid_chroms]
    val_idx = [i for i, c in enumerate(chroms) if c in valid_chroms]

    if len(val_idx) == 0:
        n = len(dataset)
        train_size = int(0.8 * n)
        perm = np.random.permutation(n)
        train_idx = perm[:train_size].tolist()
        val_idx = perm[train_size:].tolist()

    train_set = Subset(dataset, train_idx)
    val_set = Subset(dataset, val_idx)

    labels = genomic_ds.get_labels()
    train_labels = [labels[i] for i in train_idx]
    n_pos = sum(train_labels)
    n_neg = len(train_labels) - n_pos
    pos_weight_val = n_neg / max(n_pos, 1)
    print(f"Train: {len(train_set)}, Val: {len(val_set)} "
          f"(held-out chroms: {valid_chroms})")
    print(f"Class balance -- pos: {n_pos} ({100*n_pos/len(train_labels):.1f}%), "
          f"neg: {n_neg} ({100*n_neg/len(train_labels):.1f}%), "
          f"pos_weight: {pos_weight_val:.2f}")

else:
    print(f"\n=== SYNTHETIC DATA MODE ===")
    print(f"BENGI dir '{bengi_dir}': found={os.path.isdir(bengi_dir)}")
    print(f"feats_config '{feats_config}': found={os.path.exists(feats_config)}")
    print(f"Using synthetic data for pipeline testing.")
    dataset = EPIDataset(config, encoder, num_synthetic=200)
    train_size = int(0.8 * len(dataset))
    train_set, val_set = random_split(dataset, [train_size, len(dataset)-train_size])
    pos_weight_val = 1.0
    labels = None

num_workers = config['training'].get('num_workers', 0)

# Build weighted sampler to oversample minority (positive) class
if USE_REAL_DATA:
    sample_weights = []
    for i in train_idx:
        w = pos_weight_val if labels[i] == 1 else 1.0
        sample_weights.append(w)
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_loader = DataLoader(train_set, batch_size=config['data']['batch_size'],
                              sampler=sampler, num_workers=num_workers)
    print(f"Using WeightedRandomSampler (pos oversample ~{pos_weight_val:.1f}x)")
else:
    train_loader = DataLoader(train_set, batch_size=config['data']['batch_size'],
                              shuffle=True, num_workers=num_workers)

val_loader = DataLoader(val_set, batch_size=config['data']['batch_size'],
                        num_workers=num_workers)

# 4. Model & Optimizer
model = Kansformer(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=config['training']['lr'],
                        weight_decay=config['training'].get('weight_decay', 1e-3))
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                                  patience=3)
crit_cls = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val]).to(device))
crit_reg = nn.MSELoss()
print(f"Using BCEWithLogitsLoss with pos_weight={pos_weight_val:.2f}")

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: {total_params:,} params ({train_params:,} trainable)")

# 5. Training Loop
train_loss_hist, val_loss_hist = [], []
best_val_loss = float('inf')
patience_counter = 0

use_augment = config.get('augmentation', {}).get('enabled', False)
print(f"\nStarting Training for {config['training']['epochs']} epochs...")
print(f"Augmentation: {'ON' if use_augment else 'OFF'}")
for epoch in range(config['training']['epochs']):
    model.train()
    dataset.augment = use_augment
    epoch_loss = 0

    for batch in train_loader:
        seq = batch['seq'].float().to(device)
        epi = batch['epi'].float().to(device)
        lbl = batch['label'].float().to(device)
        dst = batch['dist'].float().to(device)

        optimizer.zero_grad()
        p_cls, p_reg = model(seq, epi)

        loss = crit_cls(p_cls, lbl) + (config['training']['lambda_dist'] * crit_reg(p_reg, dst))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['training'].get('grad_clip', 1.0))
        optimizer.step()
        epoch_loss += loss.item()

    # Validation
    model.eval()
    dataset.augment = False
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            seq = batch['seq'].float().to(device)
            epi = batch['epi'].float().to(device)
            lbl = batch['label'].float().to(device)
            dst = batch['dist'].float().to(device)
            p_cls, p_reg = model(seq, epi)
            val_loss += (crit_cls(p_cls, lbl) + config['training']['lambda_dist'] * crit_reg(p_reg, dst)).item()
            all_preds.extend(torch.sigmoid(p_cls).cpu().numpy().flatten())
            all_labels.extend(lbl.cpu().numpy().flatten())

    avg_train = epoch_loss / max(len(train_loader), 1)
    avg_val = val_loss / max(len(val_loader), 1)
    train_loss_hist.append(avg_train)
    val_loss_hist.append(avg_val)

    preds_np = np.array(all_preds)
    labels_np = np.array(all_labels)
    acc = accuracy_score(labels_np, (preds_np > 0.5).astype(int))
    try:
        auc = roc_auc_score(labels_np, preds_np)
    except ValueError:
        auc = 0.0

    print(f"Epoch {epoch+1}/{config['training']['epochs']} | "
          f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
          f"Acc: {acc:.4f} | AUC: {auc:.4f}")

    scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), f"{config['paths']['save_dir']}/model_best.pth")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= config['training'].get('patience', 999):
        print(f"Early stopping at epoch {epoch+1}")
        break

# 6. Save
torch.save(model.state_dict(), f"{config['paths']['save_dir']}/model_final.pth")
plot_history(train_loss_hist, val_loss_hist, f"{config['paths']['save_dir']}/loss.png")
print("Training Complete.")

## Cell 9 — Final pre-flight check

In [ ]:
import yaml, os, json, glob

print('=' * 60)
print('FINAL PRE-FLIGHT CHECK')
print('=' * 60)
errors = []

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# feats_config
fc = cfg['paths']['feats_config']
if os.path.exists(fc):
    print(f'OK: feats_config at {fc}')
    with open(fc) as f:
        jd = json.load(f)
    cells = [k for k in jd if isinstance(jd[k], dict)]
    print(f'    Cell lines: {cells}')
else:
    errors.append(f'feats_config NOT found: {fc}')
    print(f'FAIL: {errors[-1]}')

# BENGI
tsvs = glob.glob(os.path.join(cfg['paths']['bengi_dir'], '*.tsv.gz'))
if len(tsvs) > 0:
    print(f'OK: {len(tsvs)} BENGI files')
else:
    errors.append('No BENGI files')
    print(f'FAIL: {errors[-1]}')

# ref_genome
rg = cfg['paths']['ref_genome']
if os.path.exists(rg):
    print(f'OK: ref_genome at {rg}')
else:
    errors.append(f'ref_genome NOT found: {rg}')
    print(f'FAIL: {errors[-1]}')

# model.py Sigmoid
with open('src/model.py') as f:
    if 'Sigmoid' in f.read():
        errors.append('model.py still has Sigmoid')
        print(f'FAIL: {errors[-1]}')
    else:
        print('OK: no Sigmoid in model.py')

# train.py features
with open('train.py') as f:
    tc = f.read()
for feat in ['BCEWithLogitsLoss', 'WeightedRandomSampler', 'dataset.augment', 'torch.sigmoid']:
    if feat in tc:
        print(f'OK: train.py has {feat}')
    else:
        errors.append(f'train.py missing {feat}')
        print(f'FAIL: {errors[-1]}')

# dataset.py augmentation
with open('src/dataset.py') as f:
    if '_reverse_complement' in f.read():
        print('OK: dataset.py has augmentation')
    else:
        errors.append('dataset.py missing augmentation')
        print(f'FAIL: {errors[-1]}')

# pysam
try:
    import pysam
    print(f'OK: pysam {pysam.__version__}')
except ImportError:
    errors.append('pysam not installed')
    print(f'FAIL: {errors[-1]}')

print(f"\n{'='*60}")
if errors:
    print(f'BLOCKED: {len(errors)} error(s)')
    for e in errors:
        print(f'  - {e}')
else:
    print('ALL CHECKS PASSED -- ready to train!')

## Cell 10 — Clear old checkpoints & start training

In [ ]:
import os, glob

for f in glob.glob('checkpoints/model_*.pth') + glob.glob('checkpoints/loss.png'):
    os.remove(f)
    print(f'Removed: {f}')
print('Old checkpoints cleared.')

!nohup python train.py > training_log_v3.txt 2>&1 &
import time; time.sleep(3)
!echo "PID: $(pgrep -f 'python train.py')"

## Cell 11 — Monitor training (re-run anytime)

In [ ]:
!echo '=== STARTUP (first 40 lines) ==='
!head -40 training_log_v3.txt
!echo ''
!echo '=== LATEST (last 10 lines) ==='
!tail -10 training_log_v3.txt
!echo ''
!echo '=== GPU ==='
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader